# Minimal RAG on Elasticsearch (semantic_text + hybrid search)

Same learning goal as [`minimal-rag-on-elastic-docs.ipynb`](minimal-rag-on-elastic-docs.ipynb), but **retrieval runs on your Elastic cluster** instead of Chroma + local embeddings.

**Prerequisites (you likely already did these):**
- Elastic Cloud account with a running deployment or Serverless project
- Familiarity with [Get started with vector search](https://www.elastic.co/docs/solutions/search/get-started/semantic-search) and [Semantic search with semantic_text](https://www.elastic.co/docs/solutions/search/semantic-search/semantic-search-semantic-text)

**What changes vs the Chroma notebook:**
| Chroma notebook | This notebook |
|-----------------|---------------|
| Local MiniLM embeddings | Elastic **Inference** via `semantic_text` (EIS on Cloud) |
| Chroma on disk | **Elasticsearch index** |
| Vector-only search | **Hybrid search** (keyword + semantic, merged with RRF) |

**Setup credentials (Terminal, before opening Jupyter):**

```bash
export ELASTICSEARCH_URL="https://your-deployment.es.region.cloud.es.io:443"
export ELASTICSEARCH_API_KEY="your-api-key"
```

Generate an API key in Elastic Cloud (global search → **API keys**). See [search connection details](https://www.elastic.co/docs/solutions/elasticsearch-solution-project/search-connection-details).

**License note:** CC BY-NC-ND on `docs-content` — personal learning only.

## 1. Install packages

```bash
cd learning-rag
source .venv/bin/activate   # or create a venv
pip install -r requirements-elasticsearch.txt
```

In [1]:
%pip install -q elasticsearch tqdm

You should consider upgrading via the '/Users/mohan/repos/docs-content/learning-rag/.venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2. Connect to Elastic + choose doc corpus

Set **`CORPUS_FOLDER`** the same way as the Chroma notebook. Each corpus gets its **own index name** so you do not overwrite another experiment.

In [3]:
import os
import re
from pathlib import Path

from elasticsearch import Elasticsearch


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "get-started").is_dir() and (p / "README.md").is_file():
            return p
    return (here / "..").resolve()


def corpus_to_index_name(folder: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "-", folder.lower()).strip("-")
    return f"elastic-rag-{slug}"[:220]


ELASTICSEARCH_URL = os.environ.get("ELASTICSEARCH_URL", "").strip()
ELASTICSEARCH_API_KEY = os.environ.get("ELASTICSEARCH_API_KEY", "").strip()

if not ELASTICSEARCH_URL or not ELASTICSEARCH_API_KEY:
    raise EnvironmentError(
        "Set ELASTICSEARCH_URL and ELASTICSEARCH_API_KEY in your shell before running this notebook.\n"
        "Example:\n  export ELASTICSEARCH_URL=\"https://...\"\n  export ELASTICSEARCH_API_KEY=\"...\""
    )

REPO_ROOT = find_repo_root()

# --- change these to switch corpora ---
CORPUS_FOLDER = "solutions"
MAX_FILES = 100

CORPUS_SUBDIR = (REPO_ROOT / CORPUS_FOLDER).resolve()
INDEX_NAME = corpus_to_index_name(CORPUS_FOLDER)

assert CORPUS_SUBDIR.is_dir(), f"Missing folder: {CORPUS_SUBDIR}"

es = Elasticsearch(ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY)
info = es.info()
print("Connected to:", info.get("cluster_name", "?"), "| version:", info.get("version", {}).get("number", "?"))
print("Indexing Markdown from:", CORPUS_SUBDIR)
print("Elasticsearch index:", INDEX_NAME)

Connected to: a719d742f0c344f9ba1f9be882898202 | version: 9.5.0
Indexing Markdown from: /Users/mohan/repos/docs-content/solutions
Elasticsearch index: elastic-rag-solutions


## 3. Load Markdown, strip frontmatter, chunk

Same chunking logic as the Chroma notebook — RAG quality still depends on this step.

In [4]:
def strip_yaml_frontmatter(text: str) -> str:
    text = text.lstrip("\ufeff")
    if not text.startswith("---"):
        return text
    parts = text.split("---", 2)
    if len(parts) >= 3:
        return parts[2].strip()
    return text


def chunk_text(text: str, max_chars: int = 1200, overlap: int = 200):
    text = text.strip()
    chunks = []
    i = 0
    while i < len(text):
        end = min(i + max_chars, len(text))
        if end < len(text):
            para = text.rfind("\n\n", i, end)
            if para != -1 and para > i + max_chars // 2:
                end = para
        piece = text[i:end].strip()
        if len(piece) > 80:
            chunks.append(piece)
        if end >= len(text):
            break
        i = max(i + 1, end - overlap)
    return chunks


def load_corpus(root: Path, max_files: int):
    paths = sorted(root.rglob("*.md"))[:max_files]
    docs = []
    for path in paths:
        try:
            raw = path.read_text(encoding="utf-8", errors="replace")
        except OSError:
            continue
        body = strip_yaml_frontmatter(raw)
        rel = path.relative_to(root)
        for idx, ch in enumerate(chunk_text(body)):
            docs.append(
                {
                    "id": f"{rel.as_posix()}#{idx}",
                    "document": ch,
                    "metadata": {"source": rel.as_posix()},
                }
            )
    return docs


documents = load_corpus(CORPUS_SUBDIR, MAX_FILES)
print(f"Chunks ready: {len(documents)} (from up to {MAX_FILES} .md files)")

Chunks ready: 665 (from up to 100 .md files)


## 4. Create index + bulk ingest (Elastic generates embeddings)

Mapping follows the [vector search quickstart](https://www.elastic.co/docs/solutions/search/get-started/semantic-search):
- **`content`** — plain text (keyword / BM25 search)
- **`semantic_text`** — vectors created by Elastic Inference at ingest (`copy_to`)
- **`source`** — doc path for citations

First bulk run may take several minutes while embeddings are generated.

In [6]:
from elasticsearch.helpers import bulk
from tqdm.auto import tqdm

INDEX_MAPPING = {
    "mappings": {
        "properties": {
            "semantic_text": {"type": "semantic_text"},
            "content": {"type": "text", "copy_to": "semantic_text"},
            "source": {"type": "keyword"},
        }
    }
}


def es_doc_id(raw_id: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", raw_id)[:512]


if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print("Deleted existing index:", INDEX_NAME)

es.indices.create(index=INDEX_NAME, body=INDEX_MAPPING)
print("Created index:", INDEX_NAME)


def generate_actions():
    for d in documents:
        yield {
            "_index": INDEX_NAME,
            "_id": es_doc_id(d["id"]),
            "_source": {
                "content": d["document"],
                "source": d["metadata"]["source"],
            },
        }


success, errors = bulk(es, generate_actions(), chunk_size=50, raise_on_error=False)
print(f"Indexed {success} chunks into {INDEX_NAME}")
if errors:
    print(f"Bulk errors: {len(errors)} — inspect first error:")
    print(errors[0])

es.indices.refresh(index=INDEX_NAME)
print("Index refreshed — ready to search.")

Deleted existing index: elastic-rag-solutions
Created index: elastic-rag-solutions
Indexed 665 chunks into elastic-rag-solutions
Index refreshed — ready to search.


## 5. Retrieve with hybrid search (RRF)

Combines **keyword** match on `content` and **semantic** match on `semantic_text`, then merges with [RRF](https://www.elastic.co/docs/solutions/search/hybrid-search) — the same idea as the Elastic quickstart.

Change **`QUESTION`**, re-run this cell and §6–§7. **Read top hits before trusting the answer.**

In [7]:
QUESTION = "What are the three main Elastic use cases and what should I use for each?"
# QUESTION = "How do I get started with Elastic Observability?"
# QUESTION = "What is semantic search and where do I start in the search solution?"

TOP_K = 5

search_body = {
    "size": TOP_K,
    "retriever": {
        "rrf": {
            "retrievers": [
                {
                    "standard": {
                        "query": {"match": {"content": QUESTION}}
                    }
                },
                {
                    "standard": {
                        "query": {"match": {"semantic_text": QUESTION}}
                    }
                },
            ]
        }
    },
}

response = es.search(index=INDEX_NAME, body=search_body)
hits = response["hits"]["hits"]

print("QUESTION:", QUESTION)
print("\n--- TOP CHUNKS (read these first) ---\n")
for rank, hit in enumerate(hits, start=1):
    src = hit["_source"]
    score = hit.get("_score", 0)
    doc = src.get("content", "")
    path = src.get("source", "?")
    print(f"### {rank}. {path}  (score: {score:.4f})")
    print(doc[:900] + ("…" if len(doc) > 900 else ""))
    print()

retrieved_docs = [h["_source"]["content"] for h in hits]
retrieved_sources = [h["_source"].get("source", "?") for h in hits]

QUESTION: What are the three main Elastic use cases and what should I use for each?

--- TOP CHUNKS (read these first) ---

### 1. index.md  (score: 0.0328)
# Solutions and use cases

:::{tip}
New to Elastic? Refer to [Elastic Fundamentals](/get-started/index.md) to understand the {{stack}}, its components, and your deployment options.
:::

Elastic helps you build applications for three main use cases: search, observability, and security. You can work directly with platform capabilities through APIs, use pre-built solutions with integrated UIs, or combine both approaches.

## Choose your path

| Your use case | What to use | Description |
| --- | --- | --- |
| Building search-powered applications | 1. [Core search capabilities](/solutions/search.md)<br><br> 2. [Elasticsearch solution](/solutions/elasticsearch-solution-project.md) | 1. Core {{es}} search features available across all deployment types, solutions, and project types<br><br>2. Additional UI tools that complement the core se

## 6. Build the RAG prompt (same as Chroma notebook)

The LLM still receives **plain text** — not Elasticsearch scores or vectors.

In [8]:
def build_rag_prompt(question: str, retrieved_docs: list[str], sources: list[str]) -> str:
    blocks = []
    for i, (text, src) in enumerate(zip(retrieved_docs, sources), start=1):
        blocks.append(f"[Source {i}: {src}]\n{text}")
    context = "\n\n".join(blocks)
    return (
        "You are assisting a reader of Elastic documentation. "
        "Answer ONLY using the CONTEXT below. If the context is insufficient, say what is missing. "
        "After each factual claim, add [Source N] matching the source blocks.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    )


prompt = build_rag_prompt(QUESTION, retrieved_docs, retrieved_sources)
print("--- BEGIN PROMPT (copy from here) ---")
print(prompt)
print("--- END PROMPT ---")

--- BEGIN PROMPT (copy from here) ---
You are assisting a reader of Elastic documentation. Answer ONLY using the CONTEXT below. If the context is insufficient, say what is missing. After each factual claim, add [Source N] matching the source blocks.

CONTEXT:
[Source 1: index.md]
# Solutions and use cases

:::{tip}
New to Elastic? Refer to [Elastic Fundamentals](/get-started/index.md) to understand the {{stack}}, its components, and your deployment options.
:::

Elastic helps you build applications for three main use cases: search, observability, and security. You can work directly with platform capabilities through APIs, use pre-built solutions with integrated UIs, or combine both approaches.

## Choose your path

| Your use case | What to use | Description |
| --- | --- | --- |
| Building search-powered applications | 1. [Core search capabilities](/solutions/search.md)<br><br> 2. [Elasticsearch solution](/solutions/elasticsearch-solution-project.md) | 1. Core {{es}} search features a

## 7. (Optional) Local answer with Ollama

Same optional step as the Chroma notebook — Ollama runs on your Mac; Elastic only handled retrieval.

In [9]:
import json
import urllib.error
import urllib.request

OLLAMA_MODEL = "llama3.2"

payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}
req = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    print(data.get("response", data))
except urllib.error.URLError as e:
    print("Ollama not reachable:", e)
    print("Skip this cell or install/start Ollama and pull", OLLAMA_MODEL)

According to the provided context, the three main Elastic use cases are:

1. Building search-powered applications
2. Monitoring applications or infrastructure
3. Protecting against threats

For building search-powered applications, you can use either:

* Core search capabilities available across all deployment types, solutions, and project types ([Source 1: index.md])
* Elasticsearch solution with integrated UIs ([Source 1: index.md])

For monitoring applications or infrastructure, the recommended approach is to use the Observability solution.

For protecting against threats, the recommended approach is to use the Security solution.

[Source 1: index.md]
[Source 2: elasticsearch-solution-project/get-started.md]
[Source 3: observability/apm/apm-server/use-environment-variables-in-configuration.md]
[Source 4: observability/apm/apm-server/use-environment-variables-in-configuration.md]
[Source 5: elasticsearch-solution-project/search-applications.md]


## 8. Compare Chroma vs Elastic (learning exercise)

Run the **same `QUESTION`** in both notebooks (same `CORPUS_FOLDER` + `MAX_FILES`).

| Compare | Chroma notebook | This notebook |
|---------|-----------------|---------------|
| Rank #1 source | Often vector-only | Hybrid may boost `index.md` when question words match |
| Embeddings | Local MiniLM | Elastic Inference (`semantic_text`) |
| Where data lives | `learning-rag/chroma_*` | Your Cloud index |

**Interview angle:** Hybrid search addresses cases where pure semantic retrieval ranks a tangentially related page first — you saw that with the LLM matrix page vs `solutions/index.md`.

**Cleanup:** When finished experimenting, delete the index in Kibana **Stack Management → Index Management**, or re-run §4 (it deletes and recreates `INDEX_NAME`).